# 1. numpy
NumPy（Numerical Python） 是 Python 科学计算的基础库，核心是 ndarray（多维数组）对象。
它解决的问题：Python 原生 list 做数值计算又慢又麻烦，NumPy 用 C 实现数组运算，速度快几十倍。

# 2. pandas
pandas 是 Python 的表格数据处理库，可以理解为"Python 版的 Excel + SQL 查询引擎"。它构建在 NumPy 之上，核心是两种数据结构：Series（一维带标签数组）和 DataFrame（二维带行/列标签的表格）。
它解决的问题：NumPy 数组是纯数值、无列名的同质结构，处理真实数据（混合类型、缺失值、带列名的表格、日期）很别扭；pandas 专门为这类"脏数据"设计。
主要能力：
1. 数据读取/写出
    df = pd.read_csv("data.csv")
    df = pd.read_excel("data.xlsx")
    df = pd.read_json(...) / read_sql() / read_parquet() / read_html()
    df.to_csv("out.csv", index=False)
2. 查看与筛选
    df.head()                    # 前几行
    df.info()                    # 列类型、缺失情况
    df.describe()                # 数值列统计摘要
    df[df["age"] > 26]                    # 布尔筛选
    df.loc[0, "name"]                     # 按标签取
    df.iloc[0, 1]                         # 按位置取
    df[(df["city"] == "Beijing") & (df["age"] < 30)]
    df.query("age > 26 and city == 'Beijing'")
    df.sort_values("age", ascending=False)
3. 增删改
    df["age_plus"] = df["age"] + 1        # 新增列
    df.drop(columns=["city"], inplace=True)
    df.rename(columns={"name": "姓名"})
    df["city"] = df["city"].str.upper()   # 字符串操作
4. 缺失值处理
    df.isna()                             # 缺失标记
    df.dropna()                           # 删除含缺失的行
    df.fillna(0) / df.fillna(df.mean())   # 填充
5. 分组聚合（最常用功能之一，类似SQL）
    df.groupby("city")["age"].mean()      # 按城市求平均年龄
    df.groupby("city").agg({"age": ["mean", "max"], "name": "count"})
    df.pivot_table(index="city", values="age", aggfunc="mean")
6. 合并/拼接（类似SQL JOIN）
    pd.merge(df1, df2, on="id", how="left")   # SQL join
    pd.concat([df1, df2], axis=0)             # 纵向拼接
7. 时间序列
    pd.to_datetime(df["date"])
    df.set_index("date").resample("M").sum()   # 按月重采样
    df.rolling(window=7).mean()                # 滑动窗口
8. 与numpy/sklearn互通
    df.values            # → NumPy 数组
    df.to_numpy()        # 同上，推荐
    sklearn 模型.fit(df[["a", "b"]], df["y"])   # pandas 直接喂给 sklearn

# 3. scikit-learn(sklearn)
scikit-learn 是基于 NumPy / SciPy 构建的机器学习算法库，提供传统机器学习（非深度学习）的一站式工具。
覆盖范围：
监督学习：线性/逻辑回归、SVM、决策树、随机森林、梯度提升、KNN、朴素贝叶斯
无监督学习：KMeans、DBSCAN、PCA、t-SNE、层次聚类
数据预处理：标准化、归一化、编码、缺失值填充
特征工程：特征选择、降维
模型评估与调参：交叉验证、网格搜索、混淆矩阵、ROC 曲线
流水线：Pipeline 把预处理 + 模型串起来

几乎所有模型都采用一套工作流程：
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

1. 划分数据
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
2. 建流水线：标准化 + 随机森林
model = make_pipeline(StandardScaler(), RandomForestClassifier(n_estimators=100))
3. 训练
model.fit(X_train, y_train)
4. 预测 + 评估
pred = model.predict(X_test)
print(accuracy_score(y_test, pred))

# 4.  sklearn模型基线
在进行模型学习之前，需要配置基线模型，其目的如下：
1. 定义”好“的标准，即作为参照系
例如：
| 模型     | PR-AUC | 说明 |
| Dummy    | 0.27 | 无技能线 |
| 逻辑回归 | 0.45 | 线性信息 |
| 随机森林 | 0.58 | 非线性信息 |
| XGBoost | 0.62 | 提升的边际收益只有 +0.04 |
这种情况下可发现：大部分信号是线性的，XGBoost相对随机森林只多赚了0.04，却贵了10倍
2. 及早发现数据泄露/管道bug
如果 Dummy 拿了 0.95 分，或者逻辑回归碾压 XGBoost，几乎可以断定：
     目标信息泄漏进了特征（比如把"是否结案"当特征去预测"是否欺诈"）
     训练/测试划分有问题（时间数据随机切分）
     标签构造错误
     特征里混进了 ID、时间戳
3. 判断“简单模型是否够用”
在强制要求可解释的情况下：
如果逻辑回归只比 XGBoost 低 0.02 PR-AUC，那就该上逻辑回归：上线简单、能过审、能解释、好维护
4. 防止“调参自欺”
Dummy 0.27 → +线性特征 0.18 → +非线性 0.13 → +调参 0.04 → +scale_pos_weight 
5. 确定进一步的工作方向
四种模型的分数一般是递减的边际效应
例如：
Dummy   0.27
LogReg  0.45   ← 一次性 +0.18
RF      0.58   ← +0.13
XGB     0.62   ← +0.04
到了 XGBoost 只有 +0.04 时，再想往上走，就应该去做特征工程 / 加数据 / 换问题定义，而不是继续调 max_depth

配置模型基线的要点：
| 事项 | 建议 |
|---|---|
| 顺序 | **先 Dummy → 再线性 → 再树 → 最后 XGBoost**，不要跳过或倒序 |
| 基线投入 | 基线**不要精细调参**，用合理默认值即可，它们是参照物不是选手 |
| 数据切分 | 全部在 `X_train` 上 CV 比较；`X_test` **只在最后**用一次 |
| 预处理 | 统一放进 `Pipeline`，防止标准化/编码在划分前泄漏 |
| 不平衡 | 线性用 `class_weight='balanced'`；树用    class_weight='balanced_subsample'；XGB 用 scale_pos_weight                    
| 指标 | 用 **PR-AUC / recall / F1**，不要用 accuracy 比较 |
| 记录 | 保存每级的分数与耗时，形成"模型选型表"再决策 |

# 5. 模型划分指标
例如对于一个矩阵：
|              | 预测为阳性        | 预测为阴性 |
| **实际阳性** | TP（真正例）       | FN（假负例，漏报） |
| **实际阴性** | FP（假正例，误报） | TN（真负例） |
1. Accuracy（准确率）
Accuracy = (TP+TN)/(TP+TN+FP+FN)
最直观，但只能应用于类别均衡、两类错误代价相近的情景
否则若某一方例数过多，会导致模型无需学习，只要全部猜“√”（或者“×”）就可以有很高的阳性率
2. ROC曲线与AUROC
TPR(召回率) = TP/(TP+FN)——————越大越好
FPR(假正率) = FP/(FP+TN)——————越小越好
ROC曲线：即将阈值从1滑到0，每一步算一个(FPR, TPR)点，连起来就是ROC曲线
TPR
1.0 ┤            ╭──────
    │        ╭───╯
    │    ╭───╯
    │ ╭──╯
0.0 ┼─╯─────────────────  FPR
    0                1.0
      ↑ 对角线 = 随机猜（AUC=0.5）
AUROC即ROC曲线下的面积：
0.5即对角线，表示完全随机
1.0表示完美预测，全部对
0.9以上非常优秀，0.7~0.8一般，＜0.6基本没用

注意：FPR分母为阴性数量，当阴性数量很高时，即使FP再大，FPR变化也很小，可能导致AUROC虚高
3. PR曲线与AUPRC
Precision(精确率) = TP/(TP+FP)
Recall(召回率) = TP/(TOP+FN)
同理得到PRC曲线：
Precision
1.0 ┤●
    │  ╰──╮
    │      ╰──╮
    │          ╰──╮
0.27┼─────────────●────  ← 水平虚线 = 无技能基线（= 阳性率）
    └──────────────────  Recall
    0                1.0
AUPRC即指PR曲线之下的面积，随机基线 = 阳性占比

注意：AUPRC在阳性数量较少时更加敏感，更能够反应真实价值，但不同数据集由于阳性率不一样，故不能直接横向比较AUPRC的值

# 6. KM曲线
## 1. 定义
KM曲线(Kaplan-Meier曲线)是生存分析中最常用的非参数生存曲线，用来估计：
“随着时间推移，"还没有发生事件（死亡/进展/复发）"的概率是多少”
纵轴：生存概率(0~1)
横轴：时间
曲线呈阶梯状，只有“事件发生”时刻下降
## 2. 意义
主要用于处理删失(censoring)问题
生存数据并非每个人都能看到结局，例如研究结束患者仍存活/失访/退出，则只能知道"至少活了这么久"，称为右删失
KM通过动态调整风险集解决了这个问题：
每个事件发生只看“此刻在随访的人”
删失的人在删失时刻从风险集移除，但不记为事件，曲线不下降
## 3. 如何读图
Survival probability
1.0 ┤●
    │ ╲
0.8 ┤  ●
    │   │
0.6 ┤   ●──●
    │       │
0.4 ┤       ●──────●  ← 删失标记（竖线）
    │              │
0.2 ┤              ●
    │
0.0 ┼──────────────────────  Time
    0    6   12   18   24
| 元素 | 含义 |
|---|---|
| **阶梯下降** | 每次下降 = 有人发生事件 |
| **平坦段** | 该时间段内没人发生事件 |
| **竖线小刻度** | 删失（censored）发生在该时刻，曲线不下降 |
| **中位生存期** | 从 y=0.5 画水平线，与曲线交点对应的时间 |
| **置信带** | 通常用 Greenwood 公式算 95% CI，样本越少越宽 |
| **Number at risk 表** | 图下方列出每个时间点的风险人数，**论文必备** |
## 4. 多组比较
KM曲线一般按照组别画(Responder vs Non-respnder)
两条曲线分的越开，说明组间生存差异越大
## 5. 一般的论文套路
Table 1 基线特征
   ↓
KM 曲线 + log-rank（单因素、直观展示）
   ↓
Cox 多因素（调整年龄、分期、癌种等，得到独立预后因素）
   ↓
（可选）列线图 / 风险评分 / 时间依赖 ROC

# 7. OS与PFS
## 1. OS(overall survival)
总生存期，是从治疗开始到死亡(任何原因)的时间
事件 = 死亡，无论任何原因的死亡
删失 = 研究结束仍然存活、或失访
OS是肿瘤临床试验的金标准终点
## 2. PFS(progression-free survival)
无进展生存期，是从治疗开始到疾病进展(或死亡)的时间
事件 = 进展 或 死亡(复合终点)
删失 = 最后一次影像评估时既未进展也未死亡
注： 
进展需要按标准判定(通常是 RECIST 1.1：靶病灶直径总和增加 ≥ 20% 且绝对值增加 ≥ 5 mm，或出现新病灶)
PFS是替代终点，不能代替OS

## 3. 免疫治疗中的相关问题
### 1. 假性进展(pseudo-progression)
免疫治疗初期，肿瘤可能因免疫细胞浸润而暂时"变大"，随后缩小。按传统 RECIST 会被判为进展，但患者实际获益。
需要 iRECIST 标准：首次"进展"记为 iUPD，需再次影像确认才记为 iCPD
### 2. 评估间隔影响PFS
PFS 是区间删失的：真实进展时间在两次影像之间。评估越稀疏，PFS 越"粗"。不同试验的扫描 schedule 不同，跨研究比较 PFS 要谨慎。
### 3. PFS与OS相关性在IO中较弱
免疫治疗存在"长尾生存"效应，PFS 的改善不一定线性转化为 OS 改善

## 4. 相关终点
ORR：objective response rate, 客观缓解率(CR+PR的比例)
DCR：disease control rate, 疾病控制率(CR+PR+SD)
DoR: duration of response, 缓解持续时间(从首次缓解到进展/死亡的时间)
TTP：time to progression, 到进展时间(死亡不计为事件，仅计算到进展的时间)
DFS：disease-free survival, 无病生存(根治术后到复发/死亡)
RFS：recurrence-free survival，无复发生存
EFS：event-free survival, 无事件生存

# 8. Cox模型
## 1. 定义
Cox比例风险模型是生存分析中最常用的多因素半参数回归模型，主要回答：
   在调整其他因素后，某个变量每变化一个单位，事件发生的"瞬时风险"变化多少倍？
可以在存在删失的情况下，同时估计多个变量的效应(HR)，并给出置信区间和p值
## 2. 公式
模型公式为：h(t|X) = h_0(t)· exp(β_1X_1 + β_2 X_2 + ... + β_p X_p)
h_0(t)：基线风险，即所有X=0时的风险，形状不限(非参数)
exp(β_1X_1 + β_2 X_2 + ... + β_p X_p)：风险乘子，由协变量决定(参数部分)
β: 回归系数，即模型需要估计的东西
## 3. HR(效应，或风险比)
HR = exp(β)
HR = 1，无影响
HR ＞ 1，风险增加
HR ＜ 1，风险降低
例如：
Age:  β = 0.03  → HR = exp(0.03) ≈ 1.03
解释：年龄每增加 1 岁，死亡风险增加约 3%
## 4. 重要假设前提
任意两个个体的风险比不随时间变化
检验方法：Schoenfeld 残差检验（最常用）；log(-log(S(t))) 曲线平行性目测；与时间交互项检验
如果假设不成立的处理方法：
### 1. 分层Cox
对违反假设的变量做分层(如按分期分层)，不估计其HR
### 2. 时依系数
加β(t) = β + γ·log(t) 交互项
### 3. RMST
改用限制平均生存时间差，不依赖PH
### 4. AFT模型
加速失效时间模型

# 9. C-index
## 1. 定义
C-index是生存分析版的AUC，衡量模型对"谁先发生事件"的排序能力
即：在所有"可以比较"的样本对中，模型把"更早发生事件的人"排成"更高风险"的比例

## 2. 例子
| 患者 | 时间 | 事件 | 模型风险分数 |
|---|---|---|---|
| A | 5 | 1 | 0.8 |
| B | 10 | 1 | 0.3 |
| C | 12 | 0（删失） | 0.5 |

| 对 | 是否可比 | 排序是否正确 |
|---|---|---|
| A vs B | ✅ A 更早且是事件 | A 风险 0.8 > B 0.3 → ✅ 正确 |
| A vs C | ✅ A 更早且是事件 | A 风险 0.8 > C 0.5 → ✅ 正确 |
| B vs C | ✅ B 更早且是事件 | B 风险 0.3 < C 0.5 → ❌ 错误 |

那么C = 2/3 ≈0.667

## 3. 解读
| C-index | 含义 |
|---|---|
| 0.5| 随机猜测（相当于抛硬币） |
| 0.6–0.7 | 有一定区分能力，常见于临床模型 |
| 0.7–0.8 | 较好 |
| 0.8–1.0 | 很好（肿瘤预后模型中不常见） |
| < 0.5| 比随机还差（风险方向用反了） |

## 4. 局限性
**只衡量排序，不衡量校准** 把风险分数整体乘 100，C-index 不变——它不知道"预测的绝对风险是否准确" |
**依赖删失分布** Harrell's C 在删失率高时会偏；建议同时报告 **Uno's C** |
**对"可比对"敏感** 不同研究删失模式不同，C-index 不可直接跨研究比较 |
**单一数字会掩盖时间变化** 模型可能早期好、晚期差；建议配合**时间依赖 AUC** |
**不等于临床有用性** C=0.7 的模型未必能改变决策；还需决策曲线分析（DCA） |
**测试集必须独立** 训练集上的 C-index 会偏乐观 |